In [48]:
'''
Docstring for 2501291219-05.ipynb
This notebook integrate between pipeline-01 and BigQuery input
'''

'\nDocstring for 2501291219-05.ipynb\nThis notebook integrate between pipeline-01 and BigQuery input\n'

### Import important library

In [49]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yaml
from pathlib import Path
import time
from collections import defaultdict

In [50]:
from functions.utils.logging import get_logger
from functions.utils.config  import PROJECT_ROOT, load_config
from functions.utils.llm_client import build_llm_client_from_yaml
from functions.utils.text_embeddings import GoogleEmbeddingModel
from functions.core.context_builder import build_user_context
from functions.core.history import build_history_summary

### QueryData

In [51]:
from google.cloud import bigquery

In [52]:
class DataQuery:
    def __init__(self):
        self.client = bigquery.Client()
    def get_students(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.students`
        """
        df = self.client.query(query).to_dataframe()
        return df
    def get_interactions(self):
        query = """
            SELECT *
            FROM `poc-piloturl-nonprod.gold_layer.interactions`
        """
        df = self.client.query(query).to_dataframe()
        return df 
    def get_user_events_json(self):
        query = """
        SELECT *
        FROM `poc-piloturl-nonprod.gold_layer.feeds`
        """
        df = self.client.query(query).to_dataframe()
        # ensure created_at is ISO-8601 Z format
        df["created_at"] = df["created_at"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

        feeds_lookup: Dict[str, Dict[str, Any]] = {}
        for _,row in df.iterrows():
            feed_id = row["feed_id"]
            feeds_lookup[feed_id] = {
                "feed_id"        : feed_id,
                "title"          : row["title"],
                "feed_text"      : row["feed_text"],
                "tags"           : row["tags"],                     
                "language"       : row["language"],
                "created_at"     : row["created_at"],
                "source"         : row["source"],
                "url"            : row["url"],
                "views"          : int(row["views"]),
                "embedding_input": row["embedding_input"]
            }
        return feeds_lookup
# dq = DataQuery()
# dq.get_students()   

### Cloud storage

In [53]:
import json
import numpy as np
import io
from datetime import datetime, timedelta, timezone
from google.cloud import storage

class GoogleCloudStorage:
    def __init__(self,bucket_name):
        self.client = storage.Client()
        try:
            self.bucket = self.client.get_bucket(bucket_name)
            print(f"Bucket exists  : {bucket_name}")
        except Exception:
            self.bucket = self.client.create_bucket(bucket_name, location=location)
            print(f"Bucket created : {bucket_name}")
            
    def blob_exists(self, blob_path) -> bool:
        '''check if object exists'''
        return self.bucket.blob(blob_path).exists()

    ### ---------- Upload folder function ----------- ###
    def upload_json(self,blob_path,json_data):
        '''upload json file to bucket'''
        blob   = self.bucket.blob(blob_path)
        
        blob.upload_from_string(
            json.dumps(json_data,ensure_ascii = False),
            content_type = "application/json"
        )
        print(f"uploaded JSON -> gs://{self.bucket.name}/{blob_path}")

    def upload_text(self, blob_path, text_data):
        '''upload text file to bucket'''
        blob   = self.bucket.blob(blob_path)

        blob.upload_from_string(
            text_data,
            content_type = "text/plain"
        )
        print(f"Uploaded text -> gs://{self.bucket.name}/{blob_path}")

    def upload_npy(self, blob_path, array):
        '''upload embedding vector'''
        buffer = io.BytesIO()
        np.save(buffer, array)
        buffer.seek(0)
        
        blob = self.bucket.blob(blob_path)
        blob.upload_from_file(
            buffer,
            content_type = "application/octet-stream"
        )
        print(f"Uploaded NPY -> gs://{self.bucket.name}/{blob_path}")
        
    ### ---------- Read file function ----------- ###
    def read_json(self, blob_path):
        '''read json file'''
        blob   = self.bucket.blob(blob_path)
        return json.loads(blob.download_as_text())

    def read_text(self, blob_path):
        '''read text file'''
        blob   = self.bucket.blob(blob_path)
        return blob.download_as_text()

    def read_npy(self, blob_path):
        '''read .npy (embedding vector) file'''
        blob   = self.bucket.blob(blob_path)

        buffer = io.BytesIO()
        blob.download_to_file(buffer)
        buffer.seek(0)
        return np.load(buffer)
        
    ### ---------- Creation folder function ----------- ###
    def create_folder(self,folder_path):
        '''Creating folder and sub folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blob = self.bucket.blob(folder_path)
        blob.upload_from_string("")
        print(f"Folder created : gs://{self.bucket}/{folder_path}")
        
    ### ---------- Remove function ----------- ###
    def delete_blob(self, blob_path):
        blob   = self.bucket.blob(blob_path)
        if blob.exists():
            blob.delete()
        print(f"Deleted: gs://{self.bucket_name}/{blob_path}")

    def delete_folder(self, folder_path):
        '''Remove nest blob(file) in folder'''
        if not folder_path.endswith("/"):
            folder_path += "/"
        blobs = self.bucket.list_blobs(prefix=folder_path)
        count = 0
        for blob in blobs:
            blob.delete()
            count += 1
    
        print(f"Deleted {count} objects under gs://{self.bucket_name}/{folder_path}")

    def delete_by_ttl(self, prefix, ttl: timedelta):
        '''Remove folder with setting time
        timeformat support
        timedelta(
            days=...,
            seconds=...,
            microseconds=...,
            milliseconds=...,
            minutes=...,
            hours=...,
            weeks=...
        )
        '''
        now    = datetime.now(timezone.utc)
        blobs  = bucket.list_blobs(prefix=prefix)
        deleted = 0
        for blob in blobs:
            if blob.time_created and now - blob.time_created > ttl:
                blob.delete()
                deleted += 1
        print(f"TTL cleanup deleted {deleted} objects under {prefix}")
        
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake")

Bucket exists  : hyde-datalake


### Helper function

In [54]:
def ensure_dir(path: str) -> None:
    """Create directory if it does not exist (idempotent)."""
    os.makedirs(path, exist_ok=True)
    
def _read_hyde_config(cfg: Dict[str, Any]) -> Tuple[int, int, int, bool, str]:
    """
    Read HyDE-related configuration with safe defaults.

    Returns
    -------
    history_threshold:
        Event count threshold for prompt selection
    recent_k:
        Max number of recent feeds used in HistorySummary
    feed_text_max_chars:
        Per-feed text truncation limit
    include_recent_feeds:
        Whether HistorySummary may include feed snippets
    query_embedding_model_name:
        Embedding model for HyDE queries
    """
    hyde_cfg = cfg.get("hyde", {}) if isinstance(cfg, dict) else {}

    history_threshold = int(hyde_cfg.get("history_threshold", 5))
    recent_k = int(hyde_cfg.get("recent_k", 5))
    feed_text_max_chars = int(hyde_cfg.get("feed_text_max_chars", 240))
    include_recent_feeds = bool(hyde_cfg.get("include_recent_feeds", True))

    # Default to same embedding family as feed embeddings
    query_embedding_model_name = str(
        hyde_cfg.get("query_embedding_model_name")
        or cfg.get("embeddings", {}).get("model_name", "")
        or "gemini-embedding-001"
    )

    # Hard safety guards
    history_threshold = max(1, history_threshold)
    recent_k = max(0, min(recent_k, 10))
    feed_text_max_chars = max(0, min(feed_text_max_chars, 2000))

    return (
        history_threshold,
        recent_k,
        feed_text_max_chars,
        include_recent_feeds,
        query_embedding_model_name,
    )
    
def read_jsonl(path: str) -> List[Dict[str, Any]]:
    """
    Deterministic JSONL reader.

    Order is preserved, which is critical for any downstream alignment.
    """
    rows: List[Dict[str, Any]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Invalid JSONL at line {line_no}: {e}") from e
    return rows

def load_prompts() -> Dict[str, str]:
    """
    Load HyDE prompt templates from parameters/prompts.yaml.

    Expected structure:
      hyde_prompts:
        hyde_a: "..."
        hyde_b: "..."
        hyde_c: "..."
    """
    import yaml

    prompts_path = PROJECT_ROOT / "parameters" / "prompts.yaml"
    with prompts_path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f) or {}

    return data.get("hyde_prompts", {}) or {}

# =============================================================================
# Prompt selection and rendering
# =============================================================================
def choose_hyde_prompt_key(num_events: int, history_threshold: int = 5) -> str:
    """
    Select HyDE prompt variant based on interaction volume.

    Rules
    -----
    - num_events >= history_threshold → history-heavy (hyde_b)
    - num_events <= 1               → onboarding / sparse (hyde_c)
    - otherwise                     → mixed (hyde_a)
    """
    if num_events >= history_threshold:
        return "hyde_b"
    if num_events <= 1:
        return "hyde_c"
    return "hyde_a"


def render_prompt(
    template: str,
    preferred_language: str,
    user_context_text: str,
    history_summary_text: Optional[str],
) -> str:
    """
    Render a prompt template using strict placeholder substitution.

    Supported placeholders:
    - {{preferred_language}}
    - {{UserContextText}}
    - {{HistorySummaryText}}

    No templating engine is used on purpose to keep behavior explicit.
    """
    s = template.replace("{{preferred_language}}", preferred_language or "th")
    s = s.replace("{{UserContextText}}", user_context_text or "")
    s = s.replace("{{HistorySummaryText}}", history_summary_text or "")
    return s

# =============================================================================
# HyDE output handling
# =============================================================================
def _extract_hyde_query_texts(hyde_json: Dict[str, Any]) -> List[str]:
    """
    Extract query_text values from HyDE JSON output.

    Expected structure:
      {
        "hyde_queries": [
          {"query_id": "...", "query_text": "...", ...},
          ...
        ]
      }

    Order is preserved and MUST match embedding row order.
    """
    if not isinstance(hyde_json, dict):
        raise ValueError("hyde_output must be a dict")

    items = hyde_json.get("hyde_queries") or []
    if not isinstance(items, list):
        raise ValueError("hyde_output.hyde_queries must be a list")

    out: List[str] = []
    for i, it in enumerate(items):
        if not isinstance(it, dict):
            raise ValueError(f"hyde_output.hyde_queries[{i}] must be an object")
        out.append(str(it.get("query_text") or "").strip())

    return out


def _l2_normalize_rows(x: np.ndarray) -> np.ndarray:
    """
    Row-wise L2 normalization.

    Zero rows are left as zero to avoid NaNs.
    """
    if x.ndim != 2:
        raise ValueError("Expected 2D array for row normalization")

    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return (x / norms).astype(np.float32)


def _atomic_save_npy(path: str, arr: np.ndarray) -> None:
    """
    Best-effort atomic .npy write.

    Writes to a temp file and renames to avoid partial reads.
    """
    tmp = path + ".tmp.npy"
    np.save(tmp, arr)
    os.replace(tmp, path)

<hr>

# Main

### Load resource

In [55]:
cfg = load_config()
out_dir = cfg["artifacts"]["user_query_bundles_dir"]

bq = DataQuery()

In [56]:
# students_path = cfg["data"]["students_path"]           # 'data/students.csv'
# students      = pd.read_csv(students_path)
# students
students = bq.get_students() 

/usr/local/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [57]:
# interactions_path = cfg["data"]["interactions_path"]   # 'data/interactions.csv'
# interactions = pd.read_csv(interactions_path)
# interactions
interactions = bq.get_interactions() 

In [58]:
# feeds_path = cfg["data"]["feeds_path"]                 # 'data/feeds.jsonl'
# feeds_lookup: Dict[str, Dict[str, Any]] = {}
# if os.path.exists(feeds_path):
#     feeds = read_jsonl(feeds_path)
#     feeds_lookup = {
#         str(f.get("feed_id")): f
#         for f in feeds
#         if isinstance(f, dict) and f.get("feed_id") is not None
#     }
feeds_lookup = bq.get_user_events_json()

### Read HyDE-related configuration once

In [59]:
(history_threshold,recent_k,feed_text_max_chars,include_recent_feeds,query_embedding_model_name) = _read_hyde_config(cfg)
expected_dim = int(cfg.get("embeddings", {}).get("dim", 0) or 0)

In [60]:
prompts = load_prompts()
if not prompts:
    raise ValueError("hyde_prompts missing from parameters/prompts.yaml")
client = build_llm_client_from_yaml(
    parameters_path=str(PROJECT_ROOT / "parameters" / "parameters.yaml"),
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
query_embedder = GoogleEmbeddingModel(
    model_name=query_embedding_model_name,
    credentials_path=str(PROJECT_ROOT / "parameters" / "credentials.yaml"),
)
now_iso = datetime.now(timezone.utc).replace(microsecond=0).isoformat()

In [61]:
query_embedder

GoogleEmbeddingModel(model_name='gemini-embedding-001', credentials_path='/code/src/parameters/credentials.yaml', output_dim=768, uniqueness_guard_enabled=True, uniqueness_guard_min_unique_ratio=0.85, uniqueness_guard_round_decimals=8, _client=None)

In [62]:
cgs = GoogleCloudStorage(bucket_name = "hyde-datalake-feeds")

Bucket exists  : hyde-datalake-feeds


In [63]:
verbose = 1

In [64]:
# # ------------------------------------------------------------------
# # Generate one cached bundle per student
# # ------------------------------------------------------------------
# rows = []
# logger = get_logger("pipeline_1_user_hyde")
# for _, row in students.iterrows():
#     student_row = row.to_dict()     # convert pd -> dict for each row
#     student_id  = str(student_row.get("student_id","")).strip()
#     if not student_id or student_id.lower() == "nan":
#         raise ValueError(f"Invalid student_id in students.csv: {student_row!r}")
    
#     user_ctx = build_user_context(student_row)
#     pref_lang = user_ctx.user_context_json.get("preferred_language","th")

#     user_events = interactions[interactions["user_id"] == student_id]   # <- user event from interaction.csv
#     num_events  = int(len(user_events))

#     history_summary_text : Optional[str] = None
#     #** Crate by combe data for each person student **#
#     if num_events > 0:
#         history_summary_text = build_history_summary(
#             user_events,
#             preferred_language   = pref_lang,
#             include_recent_feeds = include_recent_feeds,
#             recent_k             = recent_k,
#             feeds_lookup         = feeds_lookup or None,
#             feed_text_max_chars  = feed_text_max_chars,
#         )
    
#     prompt_key = choose_hyde_prompt_key(num_events,history_threshold)
#     template = prompts.get(prompt_key)
#     if not template:
#         raise ValueError(f"Missing prompt '{prompt_key}' in pormpts.yaml")
#     prompt = render_prompt(
#             template=template,
#             preferred_language=pref_lang,
#             user_context_text=user_ctx.user_context_text,
#             history_summary_text=history_summary_text,
#         )
#     # ------------------------------------------------------------------
#     # LLM call (JSON-only)
#     # ------------------------------------------------------------------
#     hyde_json = client.generate_json(prompt)
#     # ------------------------------------------------------------------
#     # Embed HyDE queries for fast serving
#     # ------------------------------------------------------------------
#     hyde_query_texts = _extract_hyde_query_texts(hyde_json)

#     if hyde_query_texts:
#         emb = query_embedder.embed_documents(hyde_query_texts)
#         emb = np.asarray(emb, dtype = np.float32)
#         if emb.ndim != 2:
#             raise ValueError(f"Invalid embedding shape {emb.shape}")
#         emb = _l2_normalize_rows(emb)
#         if expected_dim and emb.shape[1] != expected_dim:
#             raise ValueError(
#                 f"Embedding dim mismatch for student = {student_id}:"
#                 f"got {emb.shape[1]} expected {expected_dim}"
#             )
#         dim = int(emb.shape[1])
#     else:
#         dim = expected_dim or 0
#         emb = np.zeros((0,dim), dtype=np.float32)

#     emb_filename = f"{student_id}_hyde_q_emb.npy"
#     emb_path     = os.path.join(out_dir, emb_filename)
#     _atomic_save_npy(emb_path, emb)
#     # ---------------------------------------------------
#     # Persist cached bundle for online serving
#     # ---------------------------------------------------
#     bundle: Dict[str, Any] = {
#         "bundle_version"        : "v2_hyde_embedded_queries",
#         "student_id"            : student_id,
#         "generated_at"          : now_iso,
#         "prompt_key"            : prompt_key,
#         "preferred_language"    : pref_lang,
#         "num_events"            : num_events,
#         "user_context_json"     : user_ctx.user_context_json,
#         "user_context_text"     : user_ctx.user_context_text,
#         "history_summary_text"  : history_summary_text,
#         "hyde_output"           : hyde_json,
#         "hyde_query_embeddings" : {
#             "path"        : emb_filename,
#             "model"       : query_embedding_model_name,
#             "dim"         : dim,
#             "dtype"       : "float32",
#             "num_queries" : int(len(hyde_query_texts)),
#             "normalized"  : True,
#         },
#     }

#     out_path = os.path.join(out_dir, f"{student_id}.json")
#     with open(out_path,"w",encoding="utf-8") as f:
#         json.dump(bundle,f,ensure_ascii=False,indent=2)
#     logger.info(
#         "wrote HyDE bundle student_id=%s events=%d prompt=%s",
#         student_id,
#         num_events,
#         prompt_key,
#     )
#     if verbose > 0:
#         print(_)
#         print(f"student_row -> \n {student_row}")
#         print(f"user_ctx -> \n {user_ctx}")
#         print(f"user_events -> \n {user_events}")
#         print(f"promt_key -> {prompt_key}")
#         print(f"hyde_query_texts->{hyde_query_texts}")
#         print(f"emb -> \n{emb}")
#         print(f"emb_path ->\n{emb_path}")
#         print(f"bundle->\n{bundle}")
#         print("#"*100)
#     # for vec in emb:  # emb.shape = (N, D)
#     #     rows.append({
#     #         "user_id": student_row["student_id"],
#     #         "created_at": datetime.now(timezone.utc).isoformat(),
#     #         # "embedding": vec.tolist()
#     #         "embedding": "x"
    
#     #     })
#     # Ingest to GCS
#     cgs.create_folder(
#         folder_path = f"{student_id}/embedding/"
#     )
#     cgs.create_folder(
#         folder_path = f"{student_id}/metadata/"
#     )
#     metadata = {
#         "student_id":student_id,
#         "current_status":student_row['current_status'],
#         "education_level":student_row['education_level'],
#         "education_major":student_row['education_major'],
#         "target_roles":student_row['target_roles'],
#         "timezone":cfg["app"]["timezone"],
#         "model_name":cfg["llm"]["model_name"],
#         "max_output_tokens":cfg["llm"]["max_output_tokens"],
#         "feed_text_max_chars":cfg["hyde"]["feed_text_max_chars"],
#         "temperature":cfg["llm"]["temperature"]
#     }
#     cgs.upload_json(
#         blob_path   = f"{student_id}/metadata/metadata.json",
#         json_data   = metadata
#     )
#     cgs.upload_npy(
#         blob_path   = f"{student_id}/embedding/embedding01.npy",
#         array       = emb[0]
#     )
#     cgs.upload_npy(
#         blob_path   = f"{student_id}/embedding/embedding02.npy",
#         array       = emb[1]
#     )
#     cgs.upload_npy(
#         blob_path   = f"{student_id}/embedding/embedding03.npy",
#         array       = emb[2]
#     )
#     cgs.upload_npy(
#         blob_path   = f"{student_id}/embedding/embedding04.npy",
#         array       = emb[3]
#     )
#     cgs.upload_npy(
#         blob_path   = f"{student_id}/embedding/embedding05.npy",
#         array       = emb[4]
#     )
    
#     # break

In [66]:
times = {}
totals = defaultdict(float)
for _, row in students.iterrows():
    student_row = row.to_dict()
    student_id  = str(student_row.get("student_id","")).strip()

    student_times = {}

    # -------------------------------------------------
    # 1) BigQuery / interaction query time
    # -------------------------------------------------
    start = time.perf_counter()
    user_events = interactions[interactions["user_id"] == student_id]
    student_times["query_sec"] = time.perf_counter() - start


    # -------------------------------------------------
    # 2) history summary generation (HyDE history build)
    # -------------------------------------------------
    start = time.perf_counter()

    history_summary_text = None
    num_events = int(len(user_events))

    if num_events > 0:
        history_summary_text = build_history_summary(
            user_events,
            preferred_language=pref_lang,
            include_recent_feeds=include_recent_feeds,
            recent_k=recent_k,
            feeds_lookup=feeds_lookup or None,
            feed_text_max_chars=feed_text_max_chars,
        )

    student_times["history_sec"] = time.perf_counter() - start


    # -------------------------------------------------
    # 3) embedding time
    # -------------------------------------------------
    start = time.perf_counter()

    hyde_query_texts = _extract_hyde_query_texts(hyde_json)

    if hyde_query_texts:
        emb = query_embedder.embed_documents(hyde_query_texts)
        emb = np.asarray(emb, dtype=np.float32)
        emb = _l2_normalize_rows(emb)
    else:
        emb = np.zeros((0, expected_dim or 0), dtype=np.float32)

    student_times["embed_sec"] = time.perf_counter() - start


    # -------------------------------------------------
    # 4) metadata upload time (JSON)
    # -------------------------------------------------
    start = time.perf_counter()

    cgs.upload_json(
        blob_path=f"{student_id}/metadata/metadata.json",
        json_data=metadata
    )

    student_times["metadata_upload_sec"] = time.perf_counter() - start


    # -------------------------------------------------
    # 5) npy uploads time (ALL together)
    # -------------------------------------------------
    start = time.perf_counter()

    for idx in range(5):
        cgs.upload_npy(
            blob_path=f"{student_id}/embedding/embedding0{idx+1}.npy",
            array=emb[idx]
        )

    student_times["npy_upload_sec"] = time.perf_counter() - start


    # save
    times[student_id] = {k: round(v, 5) for k, v in student_times.items()}

    for k, v in student_times.items():
        totals[k] += v


uploaded JSON -> gs://hyde-datalake-feeds/stu_p007/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p007/embedding/embedding05.npy
uploaded JSON -> gs://hyde-datalake-feeds/stu_p006/metadata/metadata.json
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p006/embedding/embedding01.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p006/embedding/embedding02.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p006/embedding/embedding03.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p006/embedding/embedding04.npy
Uploaded NPY -> gs://hyde-datalake-feeds/stu_p006/embedding/embedding05.npy
uploaded JSON -> gs://hyde-datalake-feeds/stu_p001/metadata/metadata.json
Uploaded NPY -> gs

In [67]:
n = len(times)

print("\nPer student times:")
print(times)

print("\nAverage (seconds):")
for k, v in totals.items():
    print(f"{k}: {v/n:.5f}s")



Per student times:
{'stu_p007': {'query_sec': 0.00101, 'history_sec': 0.01339, 'embed_sec': 1.81636, 'metadata_upload_sec': 0.18499, 'npy_upload_sec': 1.23484}, 'stu_p006': {'query_sec': 0.00065, 'history_sec': 0.0101, 'embed_sec': 1.77266, 'metadata_upload_sec': 0.12241, 'npy_upload_sec': 1.28129}, 'stu_p001': {'query_sec': 0.00058, 'history_sec': 0.00979, 'embed_sec': 2.03714, 'metadata_upload_sec': 0.10613, 'npy_upload_sec': 1.28272}, 'stu_p009': {'query_sec': 0.00064, 'history_sec': 0.00971, 'embed_sec': 1.69645, 'metadata_upload_sec': 0.11913, 'npy_upload_sec': 1.16481}, 'stu_p003': {'query_sec': 0.00067, 'history_sec': 0.01111, 'embed_sec': 1.72026, 'metadata_upload_sec': 0.10847, 'npy_upload_sec': 1.15355}, 'stu_p004': {'query_sec': 0.00354, 'history_sec': 0.01051, 'embed_sec': 1.79597, 'metadata_upload_sec': 0.11897, 'npy_upload_sec': 1.1204}, 'stu_p005': {'query_sec': 0.00077, 'history_sec': 0.00888, 'embed_sec': 1.75972, 'metadata_upload_sec': 0.12841, 'npy_upload_sec': 1.15

In [69]:
times

{'stu_p007': {'query_sec': 0.00101,
  'history_sec': 0.01339,
  'embed_sec': 1.81636,
  'metadata_upload_sec': 0.18499,
  'npy_upload_sec': 1.23484},
 'stu_p006': {'query_sec': 0.00065,
  'history_sec': 0.0101,
  'embed_sec': 1.77266,
  'metadata_upload_sec': 0.12241,
  'npy_upload_sec': 1.28129},
 'stu_p001': {'query_sec': 0.00058,
  'history_sec': 0.00979,
  'embed_sec': 2.03714,
  'metadata_upload_sec': 0.10613,
  'npy_upload_sec': 1.28272},
 'stu_p009': {'query_sec': 0.00064,
  'history_sec': 0.00971,
  'embed_sec': 1.69645,
  'metadata_upload_sec': 0.11913,
  'npy_upload_sec': 1.16481},
 'stu_p003': {'query_sec': 0.00067,
  'history_sec': 0.01111,
  'embed_sec': 1.72026,
  'metadata_upload_sec': 0.10847,
  'npy_upload_sec': 1.15355},
 'stu_p004': {'query_sec': 0.00354,
  'history_sec': 0.01051,
  'embed_sec': 1.79597,
  'metadata_upload_sec': 0.11897,
  'npy_upload_sec': 1.1204},
 'stu_p005': {'query_sec': 0.00077,
  'history_sec': 0.00888,
  'embed_sec': 1.75972,
  'metadata_uplo

In [ ]:
0/0

### Got embedding

In [ ]:
# metadata = {
#     "student_id":student_id,
#     "current_status":student_row['current_status'],
#     "education_level":student_row['education_level'],
#     "education_major":student_row['education_major'],
#     "target_roles":student_row['target_roles'],
#     "timezone":cfg["app"]["timezone"],
#     "model_name":cfg["llm"]["model_name"],
#     "max_output_tokens":cfg["llm"]["max_output_tokens"],
#     "feed_text_max_chars":cfg["hyde"]["feed_text_max_chars"],
#     "temperature":cfg["llm"]["temperature"]
# }
# data

In [ ]:
# type(emb)

<hr>

In [ ]:
# CREATE TABLE `poc-piloturl-nonprod.gold_layer.user_hyde_embeddings` (
#   user_id STRING,
#   embedding ARRAY<FLOAT64>,
#   created_at TIMESTAMP
# );


In [ ]:
# row = [{
#     "user_id": student_row["student_id"],
#     "created_at": datetime.now(timezone.utc).isoformat(),
#     "embedding": emb
# }]

In [ ]:
# rows = [
#     {
#         "user_id": "U-001",
#         "hyde_signature": "abc123",
#         "embedding": [0.1, 0.2, 0.3],
#         "create_at": "2026-01-26T10:00:00Z"
#     },
#     {
#         "user_id": "U-002",
#         "hyde_signature": "def456",
#         "embedding": [0.4, 0.5, 0.6],
#         "create_at": "2026-01-26T10:05:00Z"
#     }
# ]


In [ ]:
### Upload Embedding vector to BigQuery

In [ ]:
# client = bigquery.Client()
# table_id = "poc-piloturl-nonprod.gold_layer.user_hyde_embeddings"

# # rows = []

# # for vec in emb:  # emb.shape = (N, D)
# #     rows.append({
# #         "user_id": student_row["student_id"],
# #         "embedding": vec.tolist(), 
# #         "created_at": datetime.now(timezone.utc).isoformat()
# #     })

# errors = client.insert_rows_json(table_id, rows)

# if errors:
#     raise RuntimeError(errors)

In [ ]:
# {
#     "user_id":"stu_p00x",
#     "created_at":"2026-01-29T15:51:17.728744+00:00",
#     "embedding":[0.1,-0.1,0.2,...,0.1]
# }

In [ ]:
# {
#     "user_id":"stu_p00x",
#     "created_at":"2026-01-30T09:52:00z",
#     "embedding":[0.1,-0.1,0.2,...,0.1]
# }

In [ ]:
# rows[0]

In [ ]:
    # ### ---------- Read file function ----------- ###
    # def read_json(self, blob_path):
    #     '''read json file'''
    #     blob   = self.bucket.blob(blob_path)
    #     return json.loads(blob.download_as_text())

    # def read_text(self, blob_path):
    #     '''read text file'''
    #     blob   = self.bucket.blob(blob_path)
    #     return blob.download_as_text()

    # def read_npy(self, blob_path):
    #     '''read .npy (embedding vector) file'''
    #     blob   = self.bucket.blob(blob_path)

    #     buffer = io.BytesIO()
    #     blob.download_to_file(buffer)
    #     buffer.seek(0)
    #     return np.load(buffer)

In [ ]:
from time import time

In [ ]:
start = time()
emb1  = cgs.read_npy("stu_p010/embedding/embedding01.npy")
end   = time()
print(f"Duration time : {end - start} s")

In [ ]:
start    = time()
metadata = cgs.read_json("stu_p010/metadata/metadata.json")
end      = time()
print(f"Duration time : {end - start} s")

In [ ]:
metadata

In [ ]:
emb

In [ ]:
array([[-0.0171121 ,  0.00125322,  0.0353113 , ...,  0.02181705,
         0.01287333, -0.00614728],
       [ 0.00502503, -0.00833965, -0.00744512, ...,  0.00611285,
         0.02216207, -0.02105219],
       [ 0.01023446,  0.00539919,  0.02438463, ...,  0.02341303,
         0.01760579, -0.0323468 ],
       [ 0.02810526, -0.00720979,  0.0405068 , ...,  0.00928139,
         0.01327338,  0.00367828],
       [ 0.0014535 , -0.02445278,  0.02183885, ...,  0.0051083 ,
         0.00802121, -0.0036486 ]], dtype=float32)

In [ ]:
emb.shape

In [ ]:
!tree

In [ ]:
.
├── 2501291219-05.ipynb
├── 2601301113-06.ipynb
├── __pycache__
│   └── main.cpython-312.pyc
├── artifacts
│   └── user_query_bundles
│       ├── stu_p001.json
│       ├── stu_p001_hyde_q_emb.npy

In [ ]:
hyde-datalake-feeds/
├─ stu_p001/
│  ├─ metadata/
│  │  └─ metadata.json
│  ├─ embedding/
│  │  ├─ embedding01.npy
│  │  ├─ embedding02.npy
│  │  ├─ embedding03.npy
│  │  ├─ embedding04.npy
│  │  └─ embedding05.npy
├─ stu_p002/
...
└─ stu_p010/                             